# CoLight Training on Google Colab
**Thesis:** Distributed Control of Multi-Agent Systems using Reinforcement Learning  
**Author:** Ahmed Elafandy, GUC Mechatronics Engineering  

**Before running:** Make sure you have selected a GPU runtime:  
Runtime → Change runtime type → T4 GPU

In [1]:
# ─── CELL 1: Verify GPU ───────────────────────────────────────────────────────
!nvidia-smi
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

In [7]:
# ─── CELL 2: Clone CoLight repo ──────────────────────────────────────────────
import os

GITHUB_USERNAME = 'flameplat'   # ← change this
REPO_NAME       = 'colight-Bachelor'  # ← change this

if not os.path.exists(f'/content/{REPO_NAME}'):
    !git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
else:
    print('Repo already cloned, pulling latest...')
    !cd /content/{REPO_NAME} && git pull

%cd /content/{REPO_NAME}
!ls

In [ ]:
# ─── CELL 2.5: Keep-alive (prevent Colab inactivity timeout) ─────────────────
import threading, time

def _keep_alive():
    while True:
        time.sleep(60)
        print('.', end='', flush=True)

threading.Thread(target=_keep_alive, daemon=True).start()
print("Keep-alive thread started")

In [3]:
# ─── CELL 3: Install Python dependencies ─────────────────────────────────────
# NOTE: Do NOT install tensorflow — Colab already has it with CUDA support
# Do NOT install tensorflow-macos or tensorflow-metal — those are macOS only

!pip install -q networkx scipy pandas matplotlib

# Verify keras version bundled with Colab's TF
import keras
print('Keras version:', keras.__version__)

In [4]:
# ─── CELL 4: Build CityFlow from source ───────────────────────────────────────
!apt-get install -q -y cmake build-essential python3-dev
!git submodule update --init --recursive
!pip install -q /content/{REPO_NAME}/CityFlow

# Verify
import cityflow
print('CityFlow OK')


In [5]:
# ─── CELL 5: Verify full environment ─────────────────────────────────────────
import tensorflow as tf
import keras
import cityflow
import networkx
import scipy

print('TF version    :', tf.__version__)
print('Keras version :', keras.__version__)
print('GPU available :', tf.config.list_physical_devices('GPU'))
print('CityFlow      : OK')
print('NetworkX      :', networkx.__version__)
print('SciPy         :', scipy.__version__)

In [ ]:
# ─── CELL 7a: Start checkpoint thread ────────────────────────────────────────
# Every 10 min: zips only NEW round directories and downloads them to your device.
# Each download is small (just the rounds completed since the last tick).

import threading, shutil, os, time, zipfile
from google.colab import files as colab_files

REPO       = f'/content/{REPO_NAME}'
CKPT_EVERY = 10 * 60

_ckpt_stop     = threading.Event()
_synced_rounds = set()  # tracks "memo/run_timestamp/round_X" keys already downloaded

def _do_checkpoint(label='auto'):
    ts = time.strftime('%H:%M:%S')
    records_dir = os.path.join(REPO, 'records')
    if not os.path.exists(records_dir):
        print(f"[{ts}] No records dir yet, skipping")
        return

    # Collect new round directories not yet downloaded
    new_round_paths = {}  # key -> abs path
    for memo in os.listdir(records_dir):
        memo_path = os.path.join(records_dir, memo)
        if not os.path.isdir(memo_path):
            continue
        for run_ts in os.listdir(memo_path):
            train_dir = os.path.join(memo_path, run_ts, 'train_round')
            if not os.path.isdir(train_dir):
                continue
            for round_name in os.listdir(train_dir):
                if not round_name.startswith('round_'):
                    continue
                key = f"{memo}/{run_ts}/{round_name}"
                if key not in _synced_rounds:
                    new_round_paths[key] = os.path.join(train_dir, round_name)

    if not new_round_paths:
        print(f"[{ts}] No new rounds to download")
        return

    # Zip new rounds preserving relative structure
    zip_path = f'/content/records_batch_{label}_{ts.replace(":", "")}.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for key, round_abs in new_round_paths.items():
            for root, _, files in os.walk(round_abs):
                for fname in files:
                    abs_file = os.path.join(root, fname)
                    arc_name = os.path.join(key, os.path.relpath(abs_file, round_abs))
                    zf.write(abs_file, arc_name)

    size_mb = os.path.getsize(zip_path) / 1e6
    print(f"[{ts}] Downloading {len(new_round_paths)} new rounds ({size_mb:.1f} MB)...")
    colab_files.download(zip_path)
    _synced_rounds.update(new_round_paths.keys())
    os.remove(zip_path)

def _ckpt_loop():
    while not _ckpt_stop.wait(timeout=CKPT_EVERY):
        _do_checkpoint()

_ckpt_thread = threading.Thread(target=_ckpt_loop, daemon=True)
_ckpt_thread.start()
print(f"Checkpoint thread started — downloading new rounds every {CKPT_EVERY // 60} min")

In [ ]:
# ─── CELL 7a: Start checkpoint thread ────────────────────────────────────────
# Every 10 min: zips only NEW round directories + latest replay files and downloads them.

import threading, os, time, zipfile
from google.colab import files as colab_files

REPO       = f'/content/{REPO_NAME}'
CKPT_EVERY = 10 * 60

_ckpt_stop     = threading.Event()
_synced_rounds = set()  # repo-relative round paths already downloaded

def _do_checkpoint(label='auto'):
    ts = time.strftime('%H:%M:%S')
    records_dir = os.path.join(REPO, 'records')
    if not os.path.exists(records_dir):
        print(f"[{ts}] No records dir yet, skipping")
        return

    # Collect new round directories not yet downloaded
    new_round_paths = {}
    for memo in os.listdir(records_dir):
        memo_path = os.path.join(records_dir, memo)
        if not os.path.isdir(memo_path):
            continue
        for run_ts in os.listdir(memo_path):
            train_dir = os.path.join(memo_path, run_ts, 'train_round')
            if not os.path.isdir(train_dir):
                continue
            for round_name in os.listdir(train_dir):
                if not round_name.startswith('round_'):
                    continue
                round_abs = os.path.join(train_dir, round_name)
                key = os.path.relpath(round_abs, REPO)
                if key not in _synced_rounds:
                    new_round_paths[key] = round_abs

    replays_dir = os.path.join(REPO, 'frontend', 'web', 'replays')
    has_replays = os.path.exists(replays_dir)

    if not new_round_paths and not has_replays:
        print(f"[{ts}] No new rounds or replay files to download")
        return

    zip_path = f'/content/records_batch_{label}_{ts.replace(":", "")}.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        # New round records
        for key, round_abs in new_round_paths.items():
            for root, _, files in os.walk(round_abs):
                for fname in files:
                    abs_file = os.path.join(root, fname)
                    zf.write(abs_file, os.path.relpath(abs_file, REPO))
        # Latest replay files (overwritten each round — always include current state)
        if has_replays:
            for root, _, files in os.walk(replays_dir):
                for fname in files:
                    abs_file = os.path.join(root, fname)
                    zf.write(abs_file, os.path.relpath(abs_file, REPO))

    size_mb = os.path.getsize(zip_path) / 1e6
    replay_note = ' + replay files' if has_replays else ''
    print(f"[{ts}] Downloading {len(new_round_paths)} new rounds{replay_note} ({size_mb:.1f} MB)...")
    colab_files.download(zip_path)
    _synced_rounds.update(new_round_paths.keys())
    os.remove(zip_path)

def _ckpt_loop():
    while not _ckpt_stop.wait(timeout=CKPT_EVERY):
        _do_checkpoint()

_ckpt_thread = threading.Thread(target=_ckpt_loop, daemon=True)
_ckpt_thread.start()
print(f"Checkpoint thread started — downloading new rounds + replay files every {CKPT_EVERY // 60} min")

In [ ]:
# ─── CELL 7b: RESUME — upload records batches + model checkpoint ─────────────
# Upload all records_batch_*.zip files you downloaded, plus model_checkpoint.zip.
# Then run Cell 7c to continue training.

import os, json, glob, zipfile, threading, time
from google.colab import files as colab_files

REPO       = f'/content/{REPO_NAME}'
CKPT_EVERY = 10 * 60

print("Upload your zip files (records_batch_*.zip + model_checkpoint.zip):")
uploaded = colab_files.upload()

for fname, data in uploaded.items():
    zip_path = f'/content/{fname}'
    with open(zip_path, 'wb') as f:
        f.write(data)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(REPO)
    print(f'Extracted {fname}')

# Find checkpoint.json to get resume path
ckpt_files = glob.glob(f'{REPO}/model/**/checkpoint.json', recursive=True)
if not ckpt_files:
    raise RuntimeError('No checkpoint.json found — make sure you uploaded model_checkpoint.zip')

ckpt_path   = max(ckpt_files, key=os.path.getmtime)
ckpt        = json.load(open(ckpt_path))
RESUME_PATH = ckpt['model_dir']
last_round  = ckpt['last_completed_round']
print(f'Found checkpoint: round {last_round}, will resume from round {last_round + 1}')

# Mark already-restored rounds as synced so they aren't re-downloaded
_synced_rounds = set()
records_dir = os.path.join(REPO, 'records')
for memo in os.listdir(records_dir):
    for run_ts in os.listdir(os.path.join(records_dir, memo)):
        train_dir = os.path.join(records_dir, memo, run_ts, 'train_round')
        if not os.path.isdir(train_dir):
            continue
        for round_name in os.listdir(train_dir):
            if round_name.startswith('round_'):
                _synced_rounds.add(f"{memo}/{run_ts}/{round_name}")
print(f"Marked {len(_synced_rounds)} restored rounds as already synced")

# Restart checkpoint thread
_ckpt_stop = threading.Event()

def _do_checkpoint(label='auto'):
    ts = time.strftime('%H:%M:%S')
    new_round_paths = {}
    for memo in os.listdir(records_dir):
        memo_path = os.path.join(records_dir, memo)
        if not os.path.isdir(memo_path):
            continue
        for run_ts in os.listdir(memo_path):
            train_dir = os.path.join(memo_path, run_ts, 'train_round')
            if not os.path.isdir(train_dir):
                continue
            for round_name in os.listdir(train_dir):
                if not round_name.startswith('round_'):
                    continue
                key = f"{memo}/{run_ts}/{round_name}"
                if key not in _synced_rounds:
                    new_round_paths[key] = os.path.join(train_dir, round_name)
    if not new_round_paths:
        print(f"[{ts}] No new rounds to download")
        return
    zip_path = f'/content/records_batch_{label}_{ts.replace(":", "")}.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for key, round_abs in new_round_paths.items():
            for root, _, files in os.walk(round_abs):
                for fname2 in files:
                    abs_file = os.path.join(root, fname2)
                    zf.write(abs_file, os.path.join(key, os.path.relpath(abs_file, round_abs)))
    size_mb = os.path.getsize(zip_path) / 1e6
    print(f"[{ts}] Downloading {len(new_round_paths)} new rounds ({size_mb:.1f} MB)...")
    colab_files.download(zip_path)
    _synced_rounds.update(new_round_paths.keys())
    os.remove(zip_path)

def _ckpt_loop():
    while not _ckpt_stop.wait(timeout=CKPT_EVERY):
        _do_checkpoint()

_ckpt_thread = threading.Thread(target=_ckpt_loop, daemon=True)
_ckpt_thread.start()
print(f'Checkpoint thread started — downloading new rounds every {CKPT_EVERY // 60} min')

In [ ]:
# ─── CELL 7b: RESUME — upload records batches + model checkpoint ─────────────
# Upload all records_batch_*.zip files you downloaded, plus model_checkpoint.zip.
# Then run Cell 7c to continue training.

import os, json, glob, zipfile, threading, time
from google.colab import files as colab_files

REPO       = f'/content/{REPO_NAME}'
CKPT_EVERY = 10 * 60

print("Upload your zip files (records_batch_*.zip + model_checkpoint.zip):")
uploaded = colab_files.upload()

for fname, data in uploaded.items():
    zip_path = f'/content/{fname}'
    with open(zip_path, 'wb') as f:
        f.write(data)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(REPO)
    print(f'Extracted {fname}')

# Find checkpoint.json to get resume path
ckpt_files = glob.glob(f'{REPO}/model/**/checkpoint.json', recursive=True)
if not ckpt_files:
    raise RuntimeError('No checkpoint.json found — make sure you uploaded model_checkpoint.zip')

ckpt_path   = max(ckpt_files, key=os.path.getmtime)
ckpt        = json.load(open(ckpt_path))
RESUME_PATH = ckpt['model_dir']
last_round  = ckpt['last_completed_round']
print(f'Found checkpoint: round {last_round}, will resume from round {last_round + 1}')

# Mark already-restored rounds as synced so they aren't re-downloaded
_synced_rounds = set()
records_dir    = os.path.join(REPO, 'records')
for memo in os.listdir(records_dir):
    memo_path = os.path.join(records_dir, memo)
    if not os.path.isdir(memo_path):
        continue
    for run_ts in os.listdir(memo_path):
        train_dir = os.path.join(memo_path, run_ts, 'train_round')
        if not os.path.isdir(train_dir):
            continue
        for round_name in os.listdir(train_dir):
            if round_name.startswith('round_'):
                round_abs = os.path.join(train_dir, round_name)
                _synced_rounds.add(os.path.relpath(round_abs, REPO))
print(f"Marked {len(_synced_rounds)} restored rounds as already synced")

# Restart checkpoint thread
_ckpt_stop  = threading.Event()
replays_dir = os.path.join(REPO, 'frontend', 'web', 'replays')

def _do_checkpoint(label='auto'):
    ts = time.strftime('%H:%M:%S')
    if not os.path.exists(records_dir):
        print(f"[{ts}] No records dir yet, skipping")
        return
    new_round_paths = {}
    for memo in os.listdir(records_dir):
        memo_path = os.path.join(records_dir, memo)
        if not os.path.isdir(memo_path):
            continue
        for run_ts in os.listdir(memo_path):
            train_dir = os.path.join(memo_path, run_ts, 'train_round')
            if not os.path.isdir(train_dir):
                continue
            for round_name in os.listdir(train_dir):
                if not round_name.startswith('round_'):
                    continue
                round_abs = os.path.join(train_dir, round_name)
                key = os.path.relpath(round_abs, REPO)
                if key not in _synced_rounds:
                    new_round_paths[key] = round_abs

    has_replays = os.path.exists(replays_dir)
    if not new_round_paths and not has_replays:
        print(f"[{ts}] No new rounds or replay files to download")
        return

    zip_path = f'/content/records_batch_{label}_{ts.replace(":", "")}.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for key, round_abs in new_round_paths.items():
            for root, _, files in os.walk(round_abs):
                for fname2 in files:
                    abs_file = os.path.join(root, fname2)
                    zf.write(abs_file, os.path.relpath(abs_file, REPO))
        if has_replays:
            for root, _, files in os.walk(replays_dir):
                for fname2 in files:
                    abs_file = os.path.join(root, fname2)
                    zf.write(abs_file, os.path.relpath(abs_file, REPO))

    size_mb = os.path.getsize(zip_path) / 1e6
    replay_note = ' + replay files' if has_replays else ''
    print(f"[{ts}] Downloading {len(new_round_paths)} new rounds{replay_note} ({size_mb:.1f} MB)...")
    colab_files.download(zip_path)
    _synced_rounds.update(new_round_paths.keys())
    os.remove(zip_path)

def _ckpt_loop():
    while not _ckpt_stop.wait(timeout=CKPT_EVERY):
        _do_checkpoint()

_ckpt_thread = threading.Thread(target=_ckpt_loop, daemon=True)
_ckpt_thread.start()
print(f'Checkpoint thread started — downloading new rounds + replay files every {CKPT_EVERY // 60} min')

In [ ]:
# ─── CELL 7d: Download model weights now ─────────────────────────────────────
# Run any time to download the current model/ directory (needed for resume).

import shutil
from google.colab import files

shutil.make_archive('/content/model_checkpoint', 'zip', f'/content/{REPO_NAME}', 'model')
files.download('/content/model_checkpoint.zip')
print("Downloading model_checkpoint.zip...")

In [ ]:
# ─── CELL 9: Analyze run + download plots ────────────────────────────────────
# Edit RUN_DIR to the path of your run inside records/.
# Plots are saved locally then zipped for download — no Drive needed.

import os, shutil
from google.colab import files

REPO    = f'/content/{REPO_NAME}'
RUN_DIR = f'{REPO}/records/topology_colight_6_6_900_grid/anon_6_6_900_0.3_turn_drain.json_06_04_12_25_01'  # ← change this
STEP    = 5  # sample every N rounds

%cd /content/{REPO_NAME}

!python Utility/analyze_run_colab.py {RUN_DIR} --step {STEP}

run_name = os.path.basename(os.path.normpath(RUN_DIR))
zip_path = f'/content/{run_name}_analysis'
shutil.make_archive(zip_path, 'zip', f'{REPO}/analysis_output/{run_name}')
files.download(f'{zip_path}.zip')
print(f'Downloaded {run_name}_analysis.zip')

In [ ]:
# ─── CELL 8: Download all results ────────────────────────────────────────────
# Run after training to download records, model weights, and replay files.
# Extract the zip next to your local colight-Bachelor repo to match paths.

import os, zipfile
from google.colab import files as colab_files

REPO = f'/content/{REPO_NAME}'

zip_path = '/content/colight_all_results.zip'
folders  = ['records', 'model', 'frontend/web/replays']

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in folders:
        src = os.path.join(REPO, folder)
        if not os.path.exists(src):
            print(f'WARNING: {folder}/ not found, skipping')
            continue
        for root, _, fnames in os.walk(src):
            for fname in fnames:
                abs_file = os.path.join(root, fname)
                zf.write(abs_file, os.path.relpath(abs_file, REPO))
        print(f'Added {folder}/')

size_mb = os.path.getsize(zip_path) / 1e6
print(f'Downloading colight_all_results.zip ({size_mb:.1f} MB)...')
colab_files.download(zip_path)